# Extract Yelp Dataset from Kaggle

Downloads the Yelp academic dataset from Kaggle and lands the raw files into the `yelp_poc_uc.bronze.landing_zone` Unity Catalog volume.

In [0]:
%pip install kagglehub python-dotenv --quiet
dbutils.library.restartPython()

> **Why `restartPython()`?** — Ensures newly installed packages are loaded cleanly into the interpreter. Without this, cached module states from previous runs can cause subtle import conflicts.

> **Secure by design** — The token is loaded from a `.env` file (excluded from Git via `.gitignore`) and written to `~/.kaggle/access_token` with `chmod 600`. No credentials are ever hardcoded in notebook cells or committed to version control.

> **Idempotent** — `shutil.copy2()` overwrites files with the same name, so re-running this cell never creates duplicates. `os.makedirs(..., exist_ok=True)` is also safe on reruns.
>
> **Cache cleanup** — The local Kaggle cache is deleted after files are verified in the volume, freeing disk space on serverless compute where local storage is limited and shared.

In [0]:
import os
import pathlib
from dotenv import load_dotenv

# -----------------------------------------------------------------------
# Kaggle API Token (loaded from .env file at the project root)
#
# The .env file is excluded from version control via .gitignore.
# Token format: KGAT_<hash>  (new Kaggle API token format)
#
# SETUP: Open the .env file in the project root and paste your token:
#   KAGGLE_API_TOKEN=KGAT_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
# -----------------------------------------------------------------------

# Load variables from the project-root .env file
project_root = os.path.dirname(os.getcwd())
env_path = os.path.join(project_root, ".env")
load_dotenv(env_path, override=True)

kaggle_token = os.environ.get("KAGGLE_API_TOKEN")
if not kaggle_token or kaggle_token.startswith("<"):
    raise ValueError(
        f"KAGGLE_API_TOKEN not set. Open {env_path} and paste your KGAT_* token."
    )

# Write to ~/.kaggle/access_token (fallback for CLI & older libs)
kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token)
access_token_path.chmod(0o600)

print("Kaggle API token loaded securely from .env file.")
print(f"  - KAGGLE_API_TOKEN env var: set")
print(f"  - {access_token_path}: written (chmod 600)")

In [0]:
import kagglehub
import shutil

# --------------------------------------------------
# 1. Download the Yelp dataset to a local cache
# --------------------------------------------------
local_path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
print(f"Dataset downloaded to local cache: {local_path}")

# --------------------------------------------------
# 2. Copy files into the Bronze landing_zone volume
# --------------------------------------------------
volume_path = "/Volumes/yelp_poc_uc/bronze/landing_zone/yelp_textual_dataset"
os.makedirs(volume_path, exist_ok=True)

for file_name in os.listdir(local_path):
    src = os.path.join(local_path, file_name)
    if os.path.isfile(src):
        dst = os.path.join(volume_path, file_name)
        print(f"Copying {file_name} ...")
        shutil.copy2(src, dst)

# --------------------------------------------------
# 3. Verify the landed files
# --------------------------------------------------
print("\nFiles landed in the Bronze volume:")
for f in sorted(os.listdir(volume_path)):
    size_mb = os.path.getsize(os.path.join(volume_path, f)) / (1024 * 1024)
    print(f"  {f:50s} {size_mb:>8.1f} MB")

# --------------------------------------------------
# 4. Clean up the local cache to free disk space
#    (best practice on serverless compute with
#    limited local storage — data is safely in the
#    UC volume, so the local copy is redundant)
# --------------------------------------------------
shutil.rmtree(local_path, ignore_errors=True)
print(f"\nLocal cache cleaned up: {local_path}")